# Two Learning Paradigms

The methods in this work span two fundamental machine learning paradigms, each with distinct advantages and use cases. Understanding these paradigms is crucial for selecting the right approach for specific electromagnetic design challenges.

## Supervised Learning (Chapters 2-3)

### Core Concept
Supervised learning learns from labeled examples to map inputs to outputs. In electromagnetic design, this means learning the relationship between design parameters and performance metrics from pre-computed simulation data.

### Data Requirements
**What you need:**
- **Labeled training data**: Design parameters + corresponding FEA results
- **Typical dataset size**: 1,000 - 100,000 design simulations
- **Data generation cost**: High (months of FEA computation)
- **Storage requirements**: Moderate (GBs of simulation results)

### Learning Objective
**Minimize prediction error on training data**
\[
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \|f(x_i) - y_i\|^2
\]

Where:
- $x_i$: Design parameters (geometry, materials, windings)
- $y_i$: Performance metrics (efficiency, torque, field distributions)
- $f(x_i)$: Neural network prediction
- $\mathcal{L}$: Loss function (typically MSE or MAE)

In [ ]:
# Supervised Learning Workflow
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

# Simulate electromagnetic design data
def generate_design_data(num_samples=5000):
    """Generate synthetic electromagnetic design data"""
    np.random.seed(42)
    
    # Design parameters (12 dimensions)
    # [stator_inner_radius, stator_outer_radius, rotor_radius, air_gap,
    #  magnet_thickness, magnet_width, num_slots, tooth_width, 
    #  back_iron, turns_per_phase, wire_gauge, current_density]
    X = np.random.rand(num_samples, 12)
    
    # Normalize to realistic ranges
    X[:, 0] = 20 + X[:, 0] * 30      # stator_inner_radius: 20-50 mm
    X[:, 1] = X[:, 0] + 10 + X[:, 1] * 20  # stator_outer_radius
    X[:, 2] = X[:, 0] - 2 - X[:, 2] * 3   # rotor_radius
    X[:, 3] = 0.5 + X[:, 3] * 1.5        # air_gap: 0.5-2 mm
    X[:, 4] = 2 + X[:, 4] * 4            # magnet_thickness: 2-6 mm
    X[:, 5] = 5 + X[:, 5] * 10           # magnet_width: 5-15 mm
    X[:, 6] = 12 + X[:, 6] * 24          # num_slots: 12-36
    X[:, 7] = 2 + X[:, 7] * 4            # tooth_width: 2-6 mm
    X[:, 8] = 3 + X[:, 8] * 5            # back_iron: 3-8 mm
    X[:, 9] = 10 + X[:, 9] * 50          # turns_per_phase: 10-60
    X[:, 10] = 0.5 + X[:, 10] * 1.5      # wire_gauge: 0.5-2 mm²
    X[:, 11] = 2 + X[:, 11] * 6          # current_density: 2-8 A/mm²
    
    # Performance metrics (what we want to predict)
    # Simplified physics-based relationships
    efficiency = 85 + 10 * np.exp(-X[:, 3]**2) - 0.1 * X[:, 4] + 0.05 * X[:, 8] + np.random.normal(0, 1, num_samples)
    torque = X[:, 6] * X[:, 4] * X[:, 11] * 0.1 + np.random.normal(0, 2, num_samples)
    power_factor = 0.85 + 0.1 * np.sin(X[:, 6] * np.pi / 12) - 0.02 * X[:, 3] + np.random.normal(0, 0.02, num_samples)
    
    # Stack performance metrics
    y = np.column_stack([efficiency, torque, power_factor])
    
    return X, y

# Generate data
X, y = generate_design_data(5000)
print(f"Generated {X.shape[0]} design samples")
print(f"Design parameters shape: {X.shape}")
print(f"Performance metrics shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Visualize data relationships
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Design Parameters vs Performance Metrics', fontsize=16)

# Select key parameters for visualization
key_params = [3, 4, 6]  # air_gap, magnet_thickness, num_slots
param_names = ['Air Gap (mm)', 'Magnet Thickness (mm)', 'Number of Slots']
performance_names = ['Efficiency (%)', 'Torque (Nm)', 'Power Factor']

for i, (param_idx, param_name) in enumerate(zip(key_params, param_names)):
    for j, perf_name in enumerate(performance_names):
        ax = axes[i, j]
        scatter = ax.scatter(X[:, param_idx], y[:, j], alpha=0.6, s=20, c=y[:, j], cmap='viridis')
        ax.set_xlabel(param_name)
        ax.set_ylabel(perf_name)
        ax.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax, alpha=0.7)

plt.tight_layout()
plt.show()

### Advantages of Supervised Learning

**1. Well-established theory and tools**
- Decades of research in neural network architectures
- Mature optimization algorithms (Adam, SGD, etc.)
- Extensive software ecosystem (PyTorch, TensorFlow)
- Rich literature on best practices

**2. Predictable performance within training distribution**
- Performance correlates with training data quality
- Error bounds can be estimated
- Consistent behavior across similar inputs
- Debugging methodologies are well-developed

**3. Fast inference once trained**
- Millisecond prediction times
- No iteration required for new predictions
- Scalable to large batches
- Suitable for real-time applications

In [ ]:
# Simple supervised learning model
class ElectromagneticPredictor(nn.Module):
    """Neural network for predicting motor performance"""
    
    def __init__(self, input_dim=12, output_dim=3, hidden_dims=[128, 64, 32]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Training function
def train_supervised_model(X_train, y_train, X_test, y_test, epochs=100):
    """Train supervised learning model"""
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train)
    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.FloatTensor(y_test)
    
    # Initialize model
    model = ElectromagneticPredictor()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # Training metrics
    train_losses = []
    test_losses = []
    
    print("Training supervised learning model...")
    for epoch in range(epochs):
        # Training phase
        model.train()
        optimizer.zero_grad()
        
        predictions = model(X_train_tensor)
        loss = criterion(predictions, y_train_tensor)
        
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        
        # Test phase
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                test_predictions = model(X_test_tensor)
                test_loss = criterion(test_predictions, y_test_tensor)
                test_losses.append(test_loss.item())
                
                # Calculate R² score
                ss_res = ((y_test_tensor - test_predictions) ** 2).sum()
                ss_tot = ((y_test_tensor - y_test_tensor.mean()) ** 2).sum()
                r2_score = 1 - (ss_res / ss_tot)
                
                print(f"Epoch {epoch:3d}: Train Loss = {loss.item():.4f}, "
                      f"Test Loss = {test_loss.item():.4f}, R² = {r2_score.item():.4f}")
    
    return model, train_losses, test_losses

# Train the model
model, train_losses, test_losses = train_supervised_model(X_train, y_train, X_test, y_test)

# Plot training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Training loss
ax1.plot(train_losses, label='Training Loss', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Training Loss Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Test loss (sampled every 10 epochs)
epochs_sampled = list(range(0, len(train_losses), 10))
ax2.plot(epochs_sampled, test_losses, 'r-', label='Test Loss', marker='o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Test Loss Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal training loss: {train_losses[-1]:.4f}")
print(f"Final test loss: {test_losses[-1]:.4f}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

### Challenges of Supervised Learning

**1. Requires large labeled datasets**
- **Data generation cost**: Each sample requires expensive FEA simulation
- **Time investment**: Months of computation for adequate datasets
- **Storage requirements**: Large datasets need significant storage
- **Data management**: Organizing and maintaining simulation data

**2. Limited generalization outside training distribution**
- **Performance degradation**: Poor predictions on unfamiliar designs
- **Interpolation vs. extrapolation**: Works well within bounds, fails outside
- **Coverage requirement**: Training data must span design space
- **Curse of dimensionality**: More parameters require exponentially more data

**3. Data collection bias affects model performance**
- **Sampling bias**: Non-uniform design space coverage
- **Simulation settings**: Different FEA parameters create inconsistent data
- **Quality variations**: Errors in training data propagate to model
- **Temporal drift**: Design preferences change over time

---

## Reinforcement Learning (Chapters 4-5)

### Core Concept
Reinforcement learning learns through interaction with an environment, receiving rewards for desirable actions. In electromagnetic design, an agent learns to make design decisions that optimize performance by trying different designs and getting feedback from physics simulations.

### Data Requirements
**What you need:**
- **Interactive environment**: FEA simulator that can evaluate designs
- **Reward function**: Quantifies design quality
- **No pre-computed dataset**: Data generated during training
- **Online learning**: Continuously improves from new experiences

### Learning Objective
**Maximize cumulative reward through exploration**
\[
J(\pi) = \mathbb{E}_{\tau \sim \pi}\left[\sum_{t=0}^{T} \gamma^t r_t\right]
\]

Where:
- $\pi$: Policy (design strategy)
- $\tau$: Trajectory (sequence of design decisions)
- $r_t$: Reward at time step $t$
- $\gamma$: Discount factor
- $T$: Episode length

In [ ]:
# Reinforcement Learning Environment for Design
class MotorDesignEnvironment:
    """Environment for motor design optimization using RL"""
    
    def __init__(self, target_torque=50, target_efficiency=90):
        self.target_torque = target_torque
        self.target_efficiency = target_efficiency
        self.max_steps = 20
        
        # Design parameter bounds
        self.param_bounds = {
            'air_gap': (0.5, 2.0),           # mm
            'magnet_thickness': (2.0, 6.0),  # mm
            'magnet_width': (5.0, 15.0),     # mm
            'num_slots': (12, 36),           # integer
            'tooth_width': (2.0, 6.0),       # mm
            'back_iron': (3.0, 8.0)          # mm
        }
        
        # Parameter names and order
        self.param_names = list(self.param_bounds.keys())
        self.reset()
    
    def reset(self):
        """Reset to initial random design"""
        self.current_design = {}
        self.step_count = 0
        self.design_history = []
        
        # Initialize with random values
        for param_name, (min_val, max_val) in self.param_bounds.items():
            if param_name == 'num_slots':
                self.current_design[param_name] = np.random.randint(min_val, max_val + 1)
            else:
                self.current_design[param_name] = np.random.uniform(min_val, max_val)
        
        return self.get_state()
    
    def get_state(self):
        """Get current state representation"""
        state = np.array([self.current_design[name] for name in self.param_names])
        
        # Normalize parameters to [0, 1]
        for i, name in enumerate(self.param_names):
            min_val, max_val = self.param_bounds[name]
            state[i] = (state[i] - min_val) / (max_val - min_val)
        
        return state
    
    def calculate_performance(self, design):
        """Simplified performance calculation (replaces FEA)"""
        # This is a simplified model - in practice, this would call FEA
        
        # Efficiency model (higher with optimal air gap and magnet dimensions)
        air_gap = design['air_gap']
        mag_thickness = design['magnet_thickness']
        
        efficiency = 85 + 5 * np.exp(-((air_gap - 1.0)**2) / 0.5) + \
                   2 * np.exp(-((mag_thickness - 4.0)**2) / 2.0) + \
                   np.random.normal(0, 0.5)
        
        # Torque model (higher with more slots and optimal magnet width)
        num_slots = design['num_slots']
        mag_width = design['magnet_width']
        
        torque = 0.5 * num_slots * mag_thickness * mag_width / 100 + \
                np.random.normal(0, 1.0)
        
        # Power factor model
        power_factor = 0.85 + 0.05 * np.sin(num_slots * np.pi / 12) - \
                      0.02 * air_gap + np.random.normal(0, 0.01)
        
        return {
            'efficiency': np.clip(efficiency, 70, 95),
            'torque': np.clip(torque, 10, 100),
            'power_factor': np.clip(power_factor, 0.7, 0.95)
        }
    
    def calculate_reward(self, performance):
        """Calculate reward based on design performance"""
        # Primary objectives: meet target torque and efficiency
        torque_reward = -abs(performance['torque'] - self.target_torque) / self.target_torque
        efficiency_reward = -abs(performance['efficiency'] - self.target_efficiency) / self.target_efficiency
        
        # Secondary objectives: maximize power factor
        power_factor_reward = performance['power_factor'] - 0.85
        
        # Bonus for meeting targets
        target_bonus = 0
        if performance['torque'] >= self.target_torque * 0.95:
            target_bonus += 1.0
        if performance['efficiency'] >= self.target_efficiency * 0.98:
            target_bonus += 1.0
        
        total_reward = torque_reward + efficiency_reward + power_factor_reward + target_bonus
        
        return total_reward
    
    def step(self, action):
        """Execute action and return new state, reward, done"""
        param_idx, new_value = action
        param_name = self.param_names[param_idx]
        min_val, max_val = self.param_bounds[param_idx]
        
        # Apply action (update design parameter)
        if param_name == 'num_slots':
            self.current_design[param_name] = np.clip(int(new_value * (max_val - min_val) + min_val), 
                                                     min_val, max_val)
        else:
            self.current_design[param_name] = np.clip(new_value * (max_val - min_val) + min_val, 
                                                     min_val, max_val)
        
        # Calculate performance and reward
        performance = self.calculate_performance(self.current_design)
        reward = self.calculate_reward(performance)
        
        self.step_count += 1
        self.design_history.append({
            'design': self.current_design.copy(),
            'performance': performance,
            'reward': reward
        })
        
        # Check if episode is done
        done = (self.step_count >= self.max_steps or 
                (performance['torque'] >= self.target_torque * 0.95 and 
                 performance['efficiency'] >= self.target_efficiency * 0.98))
        
        return self.get_state(), reward, done, performance

# Create environment
env = MotorDesignEnvironment()
print("Reinforcement Learning Environment for Motor Design:")
print(f"Target torque: {env.target_torque} Nm")
print(f"Target efficiency: {env.target_efficiency}%")
print(f"Design parameters: {len(env.param_names)}")
print(f"Max steps per episode: {env.max_steps}")

# Demonstrate environment
state = env.reset()
print(f"\nInitial state shape: {state.shape}")
print(f"Initial design: {env.current_design}")

# Random actions for demonstration
print("\nRunning random episode...")
for step in range(5):
    action = (np.random.randint(len(env.param_names)), np.random.random())
    new_state, reward, done, performance = env.step(action)
    
    print(f"Step {step+1}: {env.param_names[action[0]]} -> {env.current_design[env.param_names[action[0]]]:.2f}")
    print(f"  Performance: Efficiency={performance['efficiency']:.1f}%, "
          f"Torque={performance['torque']:.1f}Nm, PF={performance['power_factor']:.3f}")
    print(f"  Reward: {reward:.3f}")
    
    if done:
        break

### Advantages of Reinforcement Learning

**1. Learns directly from environment interaction**
- **No pre-computed dataset**: Generates data as needed
- **Adaptive learning**: Improves with more experience
- **Targeted exploration**: Focuses on promising design regions
- **Online adaptation**: Can adjust to changing requirements

**2. No pre-computed dataset bias**
- **Unbiased exploration**: Discovers design strategies without human bias
- **Novel solutions**: Can find non-intuitive optimal designs
- **Adaptive sampling**: Allocates computation where most valuable
- **Continuous improvement**: Keeps learning from new designs

**3. Discovers novel design strategies**
- **Emergent behaviors**: Learns design patterns humans might miss
- **Multi-objective optimization**: Naturally handles competing objectives
- **Sequential decision making**: Learns design sequences and dependencies
- **Robust policies**: Develops strategies that work across design variations

In [ ]:
# Simple RL Agent for Design
class DesignAgentRL:
    """Simple Q-learning agent for design optimization"""
    
    def __init__(self, state_dim, action_dim, learning_rate=0.1, epsilon=0.1):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.learning_rate = learning_rate
        self.epsilon = epsilon
        
        # Discretize state space for Q-table
        self.state_bins = 5  # 5 bins per dimension
        self.q_table = np.zeros((self.state_bins**state_dim, action_dim))
        
    def discretize_state(self, state):
        """Convert continuous state to discrete index"""
        discretized = np.clip((state * self.state_bins).astype(int), 0, self.state_bins - 1)
        
        # Convert multi-dimensional index to single index
        index = 0
        for i, val in enumerate(discretized):
            index += val * (self.state_bins ** i)
        
        return index
    
    def select_action(self, state, training=True):
        """Select action using epsilon-greedy policy"""
        state_idx = self.discretize_state(state)
        
        if training and np.random.random() < self.epsilon:
            # Explore: random action
            return np.random.randint(self.action_dim)
        else:
            # Exploit: best known action
            return np.argmax(self.q_table[state_idx])
    
    def update(self, state, action, reward, next_state, done):
        """Update Q-value using Q-learning"""
        state_idx = self.discretize_state(state)
        next_state_idx = self.discretize_state(next_state)
        
        # Q-learning update
        if done:
            target = reward
        else:
            target = reward + 0.95 * np.max(self.q_table[next_state_idx])  # gamma = 0.95
        
        self.q_table[state_idx, action] += self.learning_rate * (target - self.q_table[state_idx, action])
    
    def decay_epsilon(self, decay_rate=0.995):
        """Decay exploration rate"""
        self.epsilon = max(0.01, self.epsilon * decay_rate)

# Training function
def train_rl_agent(env, agent, num_episodes=500):
    """Train RL agent"""
    episode_rewards = []
    episode_lengths = []
    best_design = None
    best_reward = -float('inf')
    
    print("Training RL agent...")
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        step_count = 0
        
        done = False
        while not done:
            # Select action (parameter to modify and new value)
            param_action = agent.select_action(state)
            param_idx = param_action // 10  # Which parameter
            value_action = (param_action % 10) / 9.0  # New value (normalized)
            
            # Execute action
            next_state, reward, done, performance = env.step((param_idx, value_action))
            
            # Update agent
            agent.update(state, param_action, reward, next_state, done)
            
            state = next_state
            total_reward += reward
            step_count += 1
        
        episode_rewards.append(total_reward)
        episode_lengths.append(step_count)
        
        # Track best design
        if total_reward > best_reward:
            best_reward = total_reward
            best_design = env.current_design.copy()
            best_performance = performance
        
        # Decay epsilon
        agent.decay_epsilon()
        
        # Print progress
        if episode % 50 == 0:
            avg_reward = np.mean(episode_rewards[-50:]) if len(episode_rewards) >= 50 else np.mean(episode_rewards)
            print(f"Episode {episode:3d}: Avg Reward = {avg_reward:.3f}, "
                  f"Epsilon = {agent.epsilon:.3f}, Best Reward = {best_reward:.3f}")
    
    return episode_rewards, episode_lengths, best_design, best_performance

# Train RL agent
state_dim = len(env.param_names)
action_dim = len(env.param_names) * 10  # 10 discrete actions per parameter
agent = DesignAgentRL(state_dim, action_dim)

rewards, lengths, best_design, best_performance = train_rl_agent(env, agent, num_episodes=500)

# Plot training results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Episode rewards
window_size = 20
rewards_ma = np.convolve(rewards, np.ones(window_size)/window_size, mode='valid')
ax1.plot(rewards, alpha=0.3, label='Episode Reward')
ax1.plot(range(window_size-1, len(rewards)), rewards_ma, 'r-', linewidth=2, label=f'{window_size}-episode MA')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('RL Agent Training Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Episode lengths
ax2.plot(lengths, alpha=0.7, color='green')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Episode Length')
ax2.set_title('Episode Length Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest design found:")
for param, value in best_design.items():
    print(f"  {param}: {value:.2f}")
print(f"\nBest performance:")
for metric, value in best_performance.items():
    print(f"  {metric}: {value:.2f}")
print(f"Total reward: {best_reward:.3f}")

### Challenges of Reinforcement Learning

**1. Sample inefficient**
- **Many environment interactions required**: Thousands to millions of design evaluations
- **High computational cost**: Each interaction requires FEA simulation
- **Slow convergence**: Learning can take days or weeks
- **Exploration overhead**: Many trial designs fail to meet requirements

**2. Reward engineering is non-trivial**
- **Design specification translation**: Converting engineering requirements to rewards
- **Multi-objective balancing**: Weighting competing objectives appropriately
- **Sparse rewards**: Long sequences between meaningful rewards
- **Local optima**: Reward functions can guide agent to suboptimal solutions

**3. Training can be unstable**
- **Hyperparameter sensitivity**: Performance depends heavily on learning rates, discount factors
- **Catastrophic forgetting**: Agent can forget previously learned strategies
- **Convergence issues**: May oscillate or diverge without careful tuning
- **Reproducibility**: Different random seeds can lead to very different results

---

## Complementary Strengths

### When to Use Each Approach

#### Use Supervised Learning When:
- ✅ You have existing simulation data
- ✅ You need fast predictions for many designs
- ✅ The design space is well-defined and bounded
- ✅ Input-output relationships are stable
- ✅ You have months to generate training data

#### Use Reinforcement Learning When:
- ✅ Design objectives change frequently
- ✅ You want to discover novel design strategies
- ✅ Sequential decision making is important
- ✅ You can afford many FEA evaluations
- ✅ You need adaptive design policies

### Hybrid Approaches

**1. Supervised pre-training + RL fine-tuning**
- Use supervised learning to learn basic physics
- Use reinforcement learning to optimize specific objectives
- Reduces RL training time significantly

**2. RL for exploration + SL for verification**
- Use RL to discover promising design regions
- Use SL to quickly evaluate designs in those regions
- Combines exploration speed with prediction accuracy

**3. Transfer learning between paradigms**
- Knowledge learned in one paradigm transfers to the other
- Features learned by supervised models inform RL policies
- RL discoveries can be used to improve supervised datasets

### Key Insight

**Sequential topology optimization (Chapter 4) creates a natural RL formulation**
- Design becomes a sequence of material placement decisions
- Each decision affects future design possibilities
- Naturally fits the Markov Decision Process framework
- Enables policy gradient methods for optimization

This insight bridges the gap between traditional optimization and modern reinforcement learning, showing how classical engineering problems can benefit from AI approaches.

## Summary

### Key Differences at a Glance

| Aspect | Supervised Learning | Reinforcement Learning |
|--------|-------------------|----------------------|
| **Data** | Pre-computed labels | Interactive feedback |
| **Training** | Hours (one-time) | Days (iterative) |
| **Inference** | Milliseconds | Evolves design iteratively |
| **Generalization** | Within training distribution | Adapts to new objectives |
| **Data cost** | High upfront | Low upfront, high ongoing |
| **Best for** | Prediction tasks | Optimization tasks |
| **Chapters** | 2-3 | 4-5 |

### Bottom Line

Both paradigms are valuable tools in the designer's arsenal:

- **Supervised learning** excels at **prediction** and **interpolation** within known design spaces
- **Reinforcement learning** excels at **optimization** and **discovery** of new design strategies

The choice depends on your specific needs, available resources, and design objectives. In practice, the most powerful approaches often combine both paradigms to leverage their complementary strengths.